# Immuno Compass · Notebook 13: Open the Streamlit dashboard from Colab

Run the cells **top to bottom** (Runtime → Run all). The last cell gives you a link that opens the dashboard in a new tab.

* Nothing is retrained: the app loads the saved models in `models/` and the results in `results/`.
* The dashboard stays live as long as this Colab session is running. Stop it with Runtime → Disconnect.
* To put it online permanently instead, see `app/README.md` (Streamlit Community Cloud, free).

In [ ]:
# 1 · Get the project (the repo is private, so Colab needs a GitHub token once per session)
import os, sys
from pathlib import Path
if "google.colab" in sys.modules and not Path("/content/immuno-compass").exists():
    from getpass import getpass
    token = getpass("GitHub token (github.com/settings/tokens → fine-grained, immuno-compass, Contents: read-only): ")
    !git clone -q https://{token}@github.com/shibilshamz/immuno-compass.git /content/immuno-compass
    del token
ROOT = Path("/content/immuno-compass") if "google.colab" in sys.modules else next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "app" / "app.py").exists())
os.chdir(ROOT)
print("project folder:", ROOT)

In [ ]:
# 2 · Install the dashboard's libraries.
# scikit-learn and xgboost are pinned to the versions that saved the models, so the saved preprocessing loads correctly.
!pip install -q streamlit plotly "scikit-learn==1.9.1" "xgboost==3.2.0"

In [ ]:
# 3 · Start the dashboard in the background and wait until it is ready
import subprocess, time, urllib.request
PORT = 8501
log = open("/tmp/streamlit.log", "w")
proc = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app/app.py", "--server.port", str(PORT),
                         "--server.headless", "true", "--browser.gatherUsageStats", "false"], stdout=log, stderr=log)
for _ in range(90):
    try:
        if urllib.request.urlopen(f"http://localhost:{PORT}/_stcore/health", timeout=1).read() == b"ok":
            print("dashboard is running ✔"); break
    except Exception:
        time.sleep(1)
else:
    print("dashboard did not start; last log lines:"); print(open("/tmp/streamlit.log").read()[-2000:])

In [ ]:
# 4 · Open it. Click the link that appears below (opens in a new browser tab).
if "google.colab" in sys.modules:
    from google.colab import output
    output.serve_kernel_port_as_window(PORT)
else:
    print(f"Open http://localhost:{PORT} in your browser")

**If the link shows an error:** run cell 3 again (the first start can take ~30 s while models load), then cell 4.
**To see the dashboard inside this notebook instead of a new tab:** run `output.serve_kernel_port_as_iframe(PORT, height=900)`.